# SoRA: Sparse Low-Rank Adaptation on CoLA (GLUE)

**SoRA** extends LoRA by introducing a trainable sparse gating vector between the A and B matrices:

$$\Delta W = B \cdot \text{diag}(g) \cdot A$$

where $g$ is a sparse gate vector trained with a proximal gradient step that applies an L1/L0 penalty. During training, entries of $g$ are pushed towards zero; after training, zero-gated rank components are pruned away, yielding an **adaptively sparse** and potentially lower effective rank than standard LoRA.

**Key differences vs LoRA / AdaLoRA:**
- LoRA: fixed rank, dense $B \cdot A$
- AdaLoRA: adaptive rank via SVD decomposition + importance-based pruning of singular values
- SoRA: fixed *initial* rank, but a learned sparse gate prunes rank components implicitly via L1 penalty on $g$

**Reference:** *SoRA: Sparse Low-Rank Adaptation of Pre-trained Language Models* (Ding et al., 2023)

## 1. Install Dependencies

In [ ]:
# !pip install -q \
# transformers \
# datasets \
# evaluate \
# peft \
# accelerate \
# scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00


## 2. Imports

In [ ]:
import time
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EvalPrediction,
)
import evaluate

print("Imports done.")
print("GPU available:", torch.cuda.is_available())

Imports done.
GPU available: True


## 3. Load CoLA Dataset

In [ ]:
dataset = load_dataset("nyu-mll/glue", "cola")

print(dataset)
print("\nSample:", dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 8551
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1043
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1063
    })
})

Sample: {'sentence': "Our friends won't buy this analysis, let alone the next one we propose.", 'label': 1, 'idx': 0}


## 4. Tokenize

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(
        examples["sentence"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column("label", "labels")

# Remove columns the Trainer doesn't need
tokenized = tokenized.remove_columns(
    [col for col in tokenized["train"].column_names
     if col not in ["input_ids", "attention_mask", "token_type_ids", "labels"]]
)

# Do NOT call set_format — let the DataCollator handle tensor conversion
print("Tokenization done.")
print("Train size:", len(tokenized["train"]))
print("Validation size:", len(tokenized["validation"]))

Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

Tokenization done.
Train size: 8551
Validation size: 1043


## 5. SoRA Layer Implementation

SoRA adds a diagonal **sparse gate** $g \in \mathbb{R}^r$ between the LoRA A and B matrices:

$$h = W_0 x + \frac{\alpha}{r} B \cdot \text{diag}(g) \cdot A x$$

The gate is regularised with an **L1 penalty** during training, which encourages sparsity. After training, components where $|g_i|$ falls below a threshold are pruned, reducing the effective rank.

In [ ]:
class SoRALinear(nn.Module):
    def __init__(self, linear: nn.Linear, r: int = 8, alpha: float = 16.0):
        super().__init__()
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        in_features = linear.in_features
        out_features = linear.out_features

        self.weight = linear.weight
        self.bias = linear.bias
        self.weight.requires_grad = False
        if self.bias is not None:
            self.bias.requires_grad = False

        self.lora_A = nn.Parameter(torch.empty(r, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))
        self.gate   = nn.Parameter(torch.ones(r))

        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x):
        base_out = F.linear(x, self.weight, self.bias)
        dtype = x.dtype
        gated_A = (self.gate.unsqueeze(1) * self.lora_A).to(dtype)
        B = self.lora_B.to(dtype)
        adapter = F.linear(F.linear(x, gated_A), B)
        return base_out + self.scaling * adapter

    def get_gate_sparsity(self, threshold: float = 1e-2):
        with torch.no_grad():
            pruned = (self.gate.abs() < threshold).sum().item()
        return pruned / self.r

    def get_effective_rank(self, threshold: float = 1e-2):
        with torch.no_grad():
            return (self.gate.abs() >= threshold).sum().item()

## 6. Inject SoRA into DeBERTaV3-base

In [ ]:
def inject_sora(
    model: nn.Module,
    target_modules: list,
    r: int = 8,
    alpha: float = 16.0,
):
    """
    Replaces Linear layers whose name ends with any string in target_modules
    with SoRALinear. All other parameters are frozen.
    """
    replaced = 0
    for name, module in list(model.named_modules()):
        for target in target_modules:
            if name.endswith(target) and isinstance(module, nn.Linear):
                # Navigate to parent
                parts = name.split(".")
                parent = model
                for part in parts[:-1]:
                    parent = getattr(parent, part)
                sora_layer = SoRALinear(module, r=r, alpha=alpha)
                setattr(parent, parts[-1], sora_layer)
                replaced += 1
                break

    # Freeze everything except SoRA parameters
    for name, param in model.named_parameters():
        if not any(k in name for k in ["lora_A", "lora_B", "gate", "classifier"]):
            param.requires_grad = False

    print(f"Replaced {replaced} Linear layers with SoRALinear.")
    return model


print("inject_sora defined.")

inject_sora defined.


In [ ]:
SORA_RANK = 8
SORA_ALPHA = 16.0

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

model = inject_sora(
    base_model,
    target_modules=["query_proj", "value_proj"],
    r=SORA_RANK,
    alpha=SORA_ALPHA,
)

# Parameter count
all_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
trainable_pct = 100 * trainable_params / all_params

print(f"\nTotal parameters    : {all_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {trainable_pct:.4f}%")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias          

Replaced 24 Linear layers with SoRALinear.

Total parameters    : 184,718,786
Trainable parameters: 296,642
Trainable percentage: 0.1606%


## 7. Custom Trainer with L1 Gate Regularisation

SoRA requires an **L1 penalty on the gate vector** to drive sparsity. We subclass `Trainer` to add this term to the loss, then apply a **proximal gradient** step (soft-thresholding) after each optimiser step to hard-enforce the sparsity induced by the penalty.

In [ ]:
class SoRATrainer(Trainer):
    """
    Adds L1 regularisation on all SoRA gate vectors:
        loss_total = loss_task + lambda_l1 * sum(|g|)

    After each optimiser step, a proximal (soft-threshold) update
    is applied to each gate:
        g_i <- sign(g_i) * max(|g_i| - lr * lambda_l1, 0)
    """

    def __init__(self, *args, lambda_l1: float = 1e-3, **kwargs):
        super().__init__(*args, **kwargs)
        self.lambda_l1 = lambda_l1

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        task_loss = outputs.loss

        # L1 penalty on gates
        l1_reg = torch.tensor(0.0, device=task_loss.device)
        for module in model.modules():
            if isinstance(module, SoRALinear):
                l1_reg = l1_reg + module.gate.abs().sum()

        loss = task_loss + self.lambda_l1 * l1_reg

        return (loss, outputs) if return_outputs else loss

    def training_step(self, model, inputs, num_items_in_batch=None):
        """Standard training step followed by proximal gate update."""
        loss = super().training_step(model, inputs, num_items_in_batch)
        self._proximal_gate_update()
        return loss

    def _proximal_gate_update(self):
        """Soft-threshold gates to encourage exact zeros."""
        lr = self.args.learning_rate
        threshold = lr * self.lambda_l1
        with torch.no_grad():
            for module in self.model.modules():
                if isinstance(module, SoRALinear):
                    g = module.gate.data
                    module.gate.data = g.sign() * F.relu(g.abs() - threshold)


print("SoRATrainer defined.")

SoRATrainer defined.


## 8. Metrics

In [ ]:
metric = evaluate.load("glue", "cola")

def compute_metrics(p: EvalPrediction):
    preds = np.argmax(p.predictions, axis=1).tolist()
    labels = p.label_ids.tolist()
    return metric.compute(predictions=preds, references=labels)

print("Metrics ready.")

Metrics ready.


## 9. Training

In [ ]:
# !pip uninstall -y torchvision

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128


In [ ]:
training_args = TrainingArguments(
    output_dir="./sora_cola",
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=50,
    lr_scheduler_type="linear",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="matthews_correlation",
    logging_steps=50,
    report_to="none",
    fp16=False,       # disabled — causes dtype mismatch with custom SoRALinear
    bf16=False,       # also keep off for T4; T4 doesn't officially support bf16
)

from transformers import DataCollatorWithPadding

trainer = SoRATrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer),
    lambda_l1=1e-3,
)

start_time = time.time()
trainer.train()
training_time = (time.time() - start_time) / 60

print(f"Training Time: {training_time:.2f} minutes")

Epoch,Training Loss,Validation Loss,Matthews Correlation
1,0.791542,0.805010,0.000000
2,0.779673,0.794568,0.000000
3,0.770551,0.778519,0.000000
4,0.767919,0.778143,0.000000
5,0.743565,0.765492,0.083592
6,0.744997,0.753964,0.116023
7,0.736521,0.761215,0.140620
8,0.719749,0.753624,0.178952
9,0.708910,0.756364,0.212787
10,0.707105,0.756085,0.193220


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

Training Time: 11.68 minutes


## 10. Evaluate on Validation Set

In [ ]:
for name, param in model.named_parameters():
    if "gate" in name.lower():
        print(
            name,
            param.min().item(),
            param.max().item(),
            param.mean().item()
        )

deberta.encoder.layer.0.attention.self.query_proj.gate 0.736798882484436 0.7386379837989807 0.73757404088974
deberta.encoder.layer.0.attention.self.value_proj.gate 0.7481817007064819 0.7767996191978455 0.7632744312286377
deberta.encoder.layer.1.attention.self.query_proj.gate 0.7378647923469543 0.7434451580047607 0.7408551573753357
deberta.encoder.layer.1.attention.self.value_proj.gate 0.7538613677024841 0.8055169582366943 0.7828221321105957
deberta.encoder.layer.2.attention.self.query_proj.gate 0.7380790710449219 0.7420037984848022 0.7401342988014221
deberta.encoder.layer.2.attention.self.value_proj.gate 0.7639071941375732 0.8820507526397705 0.8264573216438293
deberta.encoder.layer.3.attention.self.query_proj.gate 0.7403008937835693 0.8190866112709045 0.778870701789856
deberta.encoder.layer.3.attention.self.value_proj.gate 0.8325892686843872 0.887800395488739 0.8638886213302612
deberta.encoder.layer.4.attention.self.query_proj.gate 0.7355961203575134 0.7412821650505066 0.73753917217254

In [ ]:
results = trainer.evaluate()

print(results)

{'eval_loss': 0.756363570690155, 'eval_matthews_correlation': 0.21278661337708615, 'eval_runtime': 3.2447, 'eval_samples_per_second': 321.446, 'eval_steps_per_second': 10.17, 'epoch': 10.0}


## 11. Trainable Parameters & Summary

In [ ]:
all_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
trainable_pct = 100 * trainable_params / all_params

print(f"Total parameters    : {all_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {trainable_pct:.4f}%")

sora_results = {
    "Method": "SoRA",
    "MCC": results["eval_matthews_correlation"],
    "Trainable_Params": trainable_params,
    "Trainable_Percentage": trainable_pct,
    "Training_Time_Min": training_time,
}

print("\n", sora_results)

Total parameters    : 184,718,786
Trainable parameters: 296,642
Trainable percentage: 0.1606%

 {'Method': 'SoRA', 'MCC': 0.21278661337708615, 'Trainable_Params': 296642, 'Trainable_Percentage': 0.16059113770918784, 'Training_Time_Min': 11.681778625647228}


## 12. Effective Rank Analysis (Gate-Based)

SoRA's effective rank per layer is simply the number of gate components that survived sparsification (i.e., $|g_i| \geq$ threshold).  
We also compute the SVD-based effective rank of $\Delta W = B \cdot \text{diag}(g) \cdot A$ for a direct comparison with LoRA.

In [ ]:
GATE_THRESHOLD = 1e-2   # gate components below this are considered pruned
SVD_THRESHOLD_RATIO = 0.01  # for SVD-based effective rank (same as LoRA notebook)

gate_ranks = []
svd_ranks  = []

for name, module in model.named_modules():
    if isinstance(module, SoRALinear):

        # --- Gate-based effective rank ---
        gate_rank = module.get_effective_rank(threshold=GATE_THRESHOLD)
        gate_ranks.append(gate_rank)

        # --- SVD-based effective rank of ΔW ---
        with torch.no_grad():
            g = module.gate.data                   # (r,)
            A = module.lora_A.data                 # (r, in)
            B = module.lora_B.data                 # (out, r)
            delta_w = B @ (g.unsqueeze(1) * A)     # (out, in)
            s = torch.linalg.svdvals(delta_w.float())
            svd_rank = (s > SVD_THRESHOLD_RATIO * s.max()).sum().item()
            svd_ranks.append(svd_rank)

        print(
            f"{name}: "
            f"gate_rank={gate_rank}/{module.r}, "
            f"svd_eff_rank={svd_rank}"
        )

print(f"\nAverage gate-based effective rank : {np.mean(gate_ranks):.2f}")
print(f"Average SVD-based effective rank  : {np.mean(svd_ranks):.2f}")

deberta.encoder.layer.0.attention.self.query_proj: gate_rank=8/8, svd_eff_rank=8
deberta.encoder.layer.0.attention.self.value_proj: gate_rank=8/8, svd_eff_rank=8
deberta.encoder.layer.1.attention.self.query_proj: gate_rank=8/8, svd_eff_rank=8
deberta.encoder.layer.1.attention.self.value_proj: gate_rank=8/8, svd_eff_rank=8
deberta.encoder.layer.2.attention.self.query_proj: gate_rank=8/8, svd_eff_rank=8
deberta.encoder.layer.2.attention.self.value_proj: gate_rank=8/8, svd_eff_rank=8
deberta.encoder.layer.3.attention.self.query_proj: gate_rank=8/8, svd_eff_rank=8
deberta.encoder.layer.3.attention.self.value_proj: gate_rank=8/8, svd_eff_rank=8
deberta.encoder.layer.4.attention.self.query_proj: gate_rank=8/8, svd_eff_rank=8
deberta.encoder.layer.4.attention.self.value_proj: gate_rank=8/8, svd_eff_rank=8
deberta.encoder.layer.5.attention.self.query_proj: gate_rank=8/8, svd_eff_rank=8
deberta.encoder.layer.5.attention.self.value_proj: gate_rank=8/8, svd_eff_rank=8
deberta.encoder.layer.6.atte

In [ ]:
# Inspect the gate values of the first SoRA layer
for name, module in model.named_modules():
    if isinstance(module, SoRALinear):
        gate_vals = module.gate.detach().cpu().numpy()
        print(f"\nLayer: {name}")
        print("Gate values:", gate_vals)
        print(
            f"Non-zero gates (|g| >= {GATE_THRESHOLD}): ",
            (np.abs(gate_vals) >= GATE_THRESHOLD).sum()
        )
        break


Layer: deberta.encoder.layer.0.attention.self.query_proj
Gate values: [0.7381536  0.7369396  0.7367989  0.738638   0.7372525  0.73847395
 0.7372067  0.7371291 ]
Non-zero gates (|g| >= 0.01):  8


In [ ]:
# Gate sparsity distribution across all thresholds
for thr in [1e-4, 1e-3, 5e-3, 1e-2]:
    active = [
        module.get_effective_rank(threshold=thr)
        for module in model.modules()
        if isinstance(module, SoRALinear)
    ]
    print(
        f"threshold={thr:.0e} -> "
        f"avg active rank={np.mean(active):.2f}, "
        f"total active rank={sum(active)}"
    )

threshold=1e-04 -> avg active rank=8.00, total active rank=192
threshold=1e-03 -> avg active rank=8.00, total active rank=192
threshold=5e-03 -> avg active rank=8.00, total active rank=192
threshold=1e-02 -> avg active rank=8.00, total active rank=192


## 13. Final Results Summary

In [ ]:
avg_gate_rank = np.mean(gate_ranks)
avg_svd_rank  = np.mean(svd_ranks)

sora_results["Avg_Gate_Effective_Rank"] = avg_gate_rank
sora_results["Avg_SVD_Effective_Rank"]  = avg_svd_rank

print("=" * 60)
print("SoRA Final Results")
print(f"MCC                     : {sora_results['MCC']:.4f}")
print(f"Training Time           : {training_time:.2f} minutes")
print(f"Trainable Params        : {trainable_params:,}")
print(f"Trainable Percentage    : {trainable_pct:.4f}%")
print(f"Avg Gate Effective Rank : {avg_gate_rank:.2f} / {SORA_RANK}")
print(f"Avg SVD Effective Rank  : {avg_svd_rank:.2f}")
print("=" * 60)

print("\nFull results dict:")
print(sora_results)

SoRA Final Results
MCC                     : 0.2128
Training Time           : 11.68 minutes
Trainable Params        : 296,642
Trainable Percentage    : 0.1606%
Avg Gate Effective Rank : 8.00 / 8
Avg SVD Effective Rank  : 8.00

Full results dict:
{'Method': 'SoRA', 'MCC': 0.21278661337708615, 'Trainable_Params': 296642, 'Trainable_Percentage': 0.16059113770918784, 'Training_Time_Min': 11.681778625647228, 'Avg_Gate_Effective_Rank': np.float64(8.0), 'Avg_SVD_Effective_Rank': np.float64(8.0)}


## 14. Save Model

In [ ]:
trainer.save_model("./sora_cola_final")
print("Model saved to ./sora_cola_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./sora_cola_final


In [ ]:
num_gates = 0

for name, param in model.named_parameters():
    if "gate" in name.lower():

        num_gates += 1

        print(
            name,
            (param.abs() > 1e-3).sum().item(),
            "/",
            param.numel()
        )

print("Total gate tensors:", num_gates)

deberta.encoder.layer.0.attention.self.query_proj.gate 8 / 8
deberta.encoder.layer.0.attention.self.value_proj.gate 8 / 8
deberta.encoder.layer.1.attention.self.query_proj.gate 8 / 8
deberta.encoder.layer.1.attention.self.value_proj.gate 8 / 8
deberta.encoder.layer.2.attention.self.query_proj.gate 8 / 8
deberta.encoder.layer.2.attention.self.value_proj.gate 8 / 8
deberta.encoder.layer.3.attention.self.query_proj.gate 8 / 8
deberta.encoder.layer.3.attention.self.value_proj.gate 8 / 8
deberta.encoder.layer.4.attention.self.query_proj.gate 8 / 8
deberta.encoder.layer.4.attention.self.value_proj.gate 8 / 8
deberta.encoder.layer.5.attention.self.query_proj.gate 8 / 8
deberta.encoder.layer.5.attention.self.value_proj.gate 8 / 8
deberta.encoder.layer.6.attention.self.query_proj.gate 8 / 8
deberta.encoder.layer.6.attention.self.value_proj.gate 8 / 8
deberta.encoder.layer.7.attention.self.query_proj.gate 8 / 8
deberta.encoder.layer.7.attention.self.value_proj.gate 8 / 8
deberta.encoder.layer.8.